# 20_E12 - Sagittal final training clean

Cierre limpio del módulo sagital SPIDER.

Este notebook no investiga desde cero. Consolida el mejor modelo sagital multiclase agrupado ya validado en E5, copia el checkpoint final, resume métricas y genera reportes comparables con el módulo axial E10/E11.

## Objetivos

- Cargar outputs sagitales E5.
- Consolidar el checkpoint multiclase agrupado como modelo sagital final.
- Extraer métricas internas y de holdout disponibles.
- Generar CSV, JSON y Markdown final.
- Dejar el módulo sagital listo para el pipeline multiplanar.

## Decisión metodológica

El modelo sagital multiclase agrupado ya fue validado previamente. Por lo tanto, E12 funciona como notebook de consolidación final, no como una exploración nueva.


In [ ]:
# Imports, rutas y setup
import json
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 180)

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

PFI_ROOT = Path("/content/drive/MyDrive/PFI_MVP")

SPIDER_ROOT = PFI_ROOT / "data" / "SPIDER"
PREPROCESS_ROOT = PFI_ROOT / "results" / "E4_preprocesamiento"
BINARY_ROOT = PFI_ROOT / "results" / "E5_baseline_mejorado_binario"
MULTICLASS_ROOT = PFI_ROOT / "results" / "E5_multiclase_agrupado"
HOLDOUT_ROOT = PFI_ROOT / "results" / "E5_multiclase_holdout"

E12_ROOT = PFI_ROOT / "results" / "E12_sagittal_final_training_clean"
FIGURES_ROOT = PFI_ROOT / "figures"
DOCS_ROOT = PFI_ROOT / "docs"
MODELS_ROOT = PFI_ROOT / "models"

for p in [E12_ROOT, FIGURES_ROOT, DOCS_ROOT, MODELS_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

paths = {
    "SPIDER_ROOT": SPIDER_ROOT,
    "PREPROCESS_ROOT": PREPROCESS_ROOT,
    "BINARY_ROOT": BINARY_ROOT,
    "MULTICLASS_ROOT": MULTICLASS_ROOT,
    "HOLDOUT_ROOT": HOLDOUT_ROOT,
    "E12_ROOT": E12_ROOT,
}

for name, path in paths.items():
    print(name, "->", path, "| exists:", path.exists())


In [ ]:
# Inventario de archivos disponibles

def list_files(root, max_files=120):
    print("\nROOT:", root)
    if not root.exists():
        print("No existe")
        return []

    files = sorted([p for p in root.rglob("*") if p.is_file()])
    print("Cantidad archivos:", len(files))

    for p in files[:max_files]:
        print("-", p.relative_to(root))

    if len(files) > max_files:
        print("... archivos omitidos:", len(files) - max_files)

    return files


multiclass_files = list_files(MULTICLASS_ROOT)
holdout_files = list_files(HOLDOUT_ROOT)
binary_files = list_files(BINARY_ROOT, max_files=40)


In [ ]:
# Localización de archivos principales

MODEL_CANDIDATES = [
    MULTICLASS_ROOT / "E5_multiclass_unet2d_grouped_best.pt",
    MODELS_ROOT / "E5_multiclass_unet2d_grouped_best.pt",
]

REPORT_CANDIDATES = sorted(list(MULTICLASS_ROOT.rglob("*.json"))) if MULTICLASS_ROOT.exists() else []
HOLDOUT_REPORT_CANDIDATES = sorted(list(HOLDOUT_ROOT.rglob("*.json"))) if HOLDOUT_ROOT.exists() else []

model_path = next((p for p in MODEL_CANDIDATES if p.exists()), None)

print("Modelo multiclase encontrado:", model_path)

print("\nJSONs MULTICLASS_ROOT:")
for p in REPORT_CANDIDATES:
    print("-", p)

print("\nJSONs HOLDOUT_ROOT:")
for p in HOLDOUT_REPORT_CANDIDATES:
    print("-", p)

assert model_path is not None, "No encontré el checkpoint sagital multiclase agrupado E5."


In [ ]:
# Copia del checkpoint final sagital

final_model_path = MODELS_ROOT / "E12_sagittal_multiclass_final_best.pt"

if not final_model_path.exists():
    shutil.copy2(model_path, final_model_path)
    print("Checkpoint copiado a:", final_model_path)
else:
    print("Checkpoint final ya existía:", final_model_path)

print("Tamaño MB:", round(final_model_path.stat().st_size / (1024 * 1024), 2))


In [ ]:
# Inspección segura del checkpoint

try:
    import torch

    checkpoint = torch.load(final_model_path, map_location="cpu")

    if isinstance(checkpoint, dict):
        checkpoint_keys = sorted(list(checkpoint.keys()))
        print("Keys checkpoint:")
        for k in checkpoint_keys:
            print("-", k)

        checkpoint_summary = {
            "num_classes": checkpoint.get("num_classes"),
            "base_channels": checkpoint.get("base_channels"),
            "target_size": checkpoint.get("target_size"),
            "sagittal_axis": checkpoint.get("sagittal_axis"),
            "slice_strategy": checkpoint.get("slice_strategy"),
            "val_dice_macro_no_bg": checkpoint.get("val_dice_macro_no_bg"),
            "val_iou_macro_no_bg": checkpoint.get("val_iou_macro_no_bg"),
        }
    else:
        checkpoint_summary = {"checkpoint_type": str(type(checkpoint))}

except Exception as exc:
    checkpoint_summary = {"error_loading_checkpoint": repr(exc)}

# Convertimos potenciales numpy/tensors simples a tipos serializables.
def make_json_safe(obj):
    if isinstance(obj, dict):
        return {str(k): make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [make_json_safe(v) for v in obj]
    if hasattr(obj, "item"):
        try:
            return obj.item()
        except Exception:
            pass
    return obj

checkpoint_summary = make_json_safe(checkpoint_summary)

print(json.dumps(checkpoint_summary, indent=2, ensure_ascii=False))

(E12_ROOT / "E12_checkpoint_summary.json").write_text(
    json.dumps(checkpoint_summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)


In [ ]:
# Carga e inventario de reportes JSON

def safe_load_json(path):
    try:
        return json.loads(Path(path).read_text(encoding="utf-8"))
    except Exception as exc:
        return {"__error__": repr(exc), "__path__": str(path)}


def flatten_json(obj, prefix=""):
    rows = []

    if isinstance(obj, dict):
        for k, v in obj.items():
            key = f"{prefix}.{k}" if prefix else str(k)
            rows.extend(flatten_json(v, key))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            key = f"{prefix}[{i}]"
            rows.extend(flatten_json(v, key))
    else:
        rows.append((prefix, obj))

    return rows


json_paths = REPORT_CANDIDATES + HOLDOUT_REPORT_CANDIDATES
json_inventory_rows = []

for jp in json_paths:
    data = safe_load_json(jp)
    flat = flatten_json(data)

    for key, value in flat:
        if any(term in key.lower() for term in ["dice", "iou", "loss", "acc", "class"]):
            json_inventory_rows.append(
                {
                    "source_file": str(jp),
                    "relative_source": str(jp.relative_to(PFI_ROOT)) if str(jp).startswith(str(PFI_ROOT)) else str(jp),
                    "key": key,
                    "value": value,
                }
            )

json_metrics_inventory = pd.DataFrame(json_inventory_rows)
json_metrics_inventory.to_csv(E12_ROOT / "E12_json_metrics_inventory.csv", index=False)

print("Métricas encontradas en JSONs:", len(json_metrics_inventory))
display(json_metrics_inventory.head(100))


In [ ]:
# Tabla consolidada de métricas sagitales

# Valores documentados previamente en la etapa sagital:
# - Multiclase agrupado validación: Dice macro sin fondo aprox 0.8392
# - Holdout multiclase: Dice macro sin fondo aprox 0.8311
# Si el checkpoint trae un valor más específico, se conserva también.

rows = []

ckpt_val_dice = checkpoint_summary.get("val_dice_macro_no_bg")
ckpt_val_iou = checkpoint_summary.get("val_iou_macro_no_bg")

if ckpt_val_dice is not None:
    rows.append(
        {
            "stage": "checkpoint_internal_validation",
            "metric": "dice_macro_no_bg",
            "value": float(ckpt_val_dice),
            "source": "E5 checkpoint",
        }
    )

if ckpt_val_iou is not None:
    rows.append(
        {
            "stage": "checkpoint_internal_validation",
            "metric": "iou_macro_no_bg",
            "value": float(ckpt_val_iou),
            "source": "E5 checkpoint",
        }
    )

# Métricas consolidadas de la documentación del proyecto.
rows.extend(
    [
        {
            "stage": "internal_validation_documented",
            "metric": "dice_macro_no_bg",
            "value": 0.8392,
            "source": "documented E5 validation",
        },
        {
            "stage": "holdout_documented",
            "metric": "dice_macro_no_bg",
            "value": 0.8311,
            "source": "documented E5 holdout",
        },
    ]
)

metrics_summary = pd.DataFrame(rows)
metrics_summary.to_csv(E12_ROOT / "E12_sagittal_metrics_summary.csv", index=False)

display(metrics_summary)


In [ ]:
# Figura simple de métricas consolidadas

plot_df = metrics_summary[metrics_summary["metric"].eq("dice_macro_no_bg")].copy()

plt.figure(figsize=(8, 4))
plt.bar(plot_df["stage"], plot_df["value"])
plt.ylabel("Dice macro sin fondo")
plt.ylim(0, 1)
plt.title("E12 - Métricas consolidadas del modelo sagital")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(FIGURES_ROOT / "E12_sagittal_dice_summary.png", dpi=150, bbox_inches="tight")
plt.show()

print("Figura guardada:", FIGURES_ROOT / "E12_sagittal_dice_summary.png")


In [ ]:
# Reporte final E12

report = {
    "notebook": "20_E12_sagittal_final_training_clean",
    "goal": "consolidate final sagittal SPIDER multiclass model",
    "strategy": "use validated E5 grouped multiclass checkpoint as final sagittal model",
    "source_model_path": str(model_path),
    "final_model_path": str(final_model_path),
    "checkpoint_summary": checkpoint_summary,
    "metrics_summary_csv": str(E12_ROOT / "E12_sagittal_metrics_summary.csv"),
    "json_metrics_inventory_csv": str(E12_ROOT / "E12_json_metrics_inventory.csv"),
    "key_documented_results": {
        "internal_val_dice_macro_no_bg": 0.8392,
        "holdout_dice_macro_no_bg": 0.8311,
    },
    "decision": "sagittal_module_consolidated_for_multiplanar_pipeline",
}

report_path = E12_ROOT / "E12_sagittal_final_report.json"
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")

md = f"""# E12 - Cierre del módulo sagital SPIDER

## Objetivo

Este documento consolida el modelo sagital multiclase agrupado entrenado y validado previamente sobre SPIDER.

## Estrategia

No se realizó una nueva investigación desde cero. Se tomó el checkpoint multiclase agrupado validado en E5 como modelo sagital final del sistema.

## Modelo final

- Checkpoint fuente: `{model_path}`
- Checkpoint final consolidado: `{final_model_path}`

## Resultados documentados

- Dice macro sin fondo en validación interna: 0.8392
- Dice macro sin fondo en holdout: 0.8311

## Decisión metodológica

El modelo sagital queda consolidado como núcleo del módulo sagital del sistema. Estos resultados son comparables con el cierre axial E10/E11 y habilitan avanzar hacia el pipeline común de inferencia multiplanar.

## Próximo paso

Construir E13: pipeline común de inferencia para sagital y axial T2.
"""

md_path = DOCS_ROOT / "E12_sagittal_final_training_clean_conclusion.md"
md_path.write_text(md, encoding="utf-8")

print("Reporte JSON:", report_path)
print("Reporte Markdown:", md_path)
print(json.dumps(report, indent=2, ensure_ascii=False))


## Salidas esperadas

Al ejecutar el notebook se generan:

```text
results/E12_sagittal_final_training_clean/
  E12_checkpoint_summary.json
  E12_json_metrics_inventory.csv
  E12_sagittal_metrics_summary.csv
  E12_sagittal_final_report.json

docs/
  E12_sagittal_final_training_clean_conclusion.md

figures/
  E12_sagittal_dice_summary.png

models/
  E12_sagittal_multiclass_final_best.pt
```

Con esto, el módulo sagital queda cerrado documentalmente y listo para pasar a E13.
